## Constants

In [ ]:
VOCAB_SIZE = 4096
SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>", "<pad>"]
UNKOWN_TOKEN = "<unk>"

CONTEXT_LENGTH = 256

NUM_PROC = 32

## Load Dataset

In [ ]:
from datasets import load_dataset

# Load the TinyStories dataset
print("Downloading TinyStories dataset...")
dataset = load_dataset("roneneldan/TinyStories")

# Let's inspect what we just downloaded
print(dataset)

# Print a sample story to see what it looks like
print("\n--- Sample Story ---")
print(dataset["train"][0]["text"])

## Generate vocabulary and create tokenizer

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Instantiate the BPE model backbone
tokenizer = Tokenizer(BPE(unk_token=UNKOWN_TOKEN))

# Pre-tokenizer to split text into words by whitespace
tokenizer.pre_tokenizer = Whitespace()

# Setup the trainer
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,                          
    special_tokens=SPECIAL_TOKENS
)

# Create a generator to stream the dataset in chunks
# This feeds the text to the tokenizer 10,000 stories at a time
def batch_iterator(batch_size=10_000):
    for i in range(0, len(dataset["train"]), batch_size):
        yield dataset["train"][i : i + batch_size]["text"]

# Train the tokenizer using the iterator instead of a text file
print("Training BPE tokenizer from dataset (this may take a minute)...")
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

# Save the trained tokenizer configuration to disk
tokenizer.save("tinystories_bpe.json")
print("BPE Tokenizer successfully trained and saved!")

### Post processing for tokenizer (not necessary, just for debuging)

In [ ]:
from tokenizers.processors import TemplateProcessing

# Load your tokenizer just to be sure
tokenizer = Tokenizer.from_file("tinystories_bpe.json")

# Configure the tokenizer to always wrap text in <bos> and <eos>
tokenizer.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    special_tokens=[
        ("<bos>", tokenizer.token_to_id("<bos>")),
        ("<eos>", tokenizer.token_to_id("<eos>")),
    ],
)

# Test it out!
test_phrase = "Lily found a little bird."
encoded = tokenizer.encode(test_phrase)

print("Tokens:", encoded.tokens)
print("IDs   :", encoded.ids)

## Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    from tokenizers import Tokenizer
    from tokenizers.processors import TemplateProcessing
    
    # Every background process loads its own copy of the tokenizer
    worker_tokenizer = Tokenizer.from_file("tinystories_bpe.json")
    
    # Re-apply the post-processor to make sure <bos> and <eos> are added by the workers
    worker_tokenizer.post_processor = TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[
            ("<bos>", worker_tokenizer.token_to_id("<bos>")),
            ("<eos>", worker_tokenizer.token_to_id("<eos>")),
        ],
    )
    
    # Process the current batch of stories
    outputs = [worker_tokenizer.encode(text).ids for text in examples["text"]]
    return {"input_ids": outputs}

print("Tokenizing the entire dataset using parallel processing...")
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=NUM_PROC,
    remove_columns=["text"] # Drops the raw strings to save RAM
)

print(tokenized_datasets)

## Group Tokens

In [ ]:
def group_texts(examples, context_length):
    # Concatenate all lists of IDs together into one massive list
    concatenated_ids = sum(examples["input_ids"], [])
    total_length = len(concatenated_ids)
    
    # Chop off the tiny remainder at the very end so it divides perfectly
    total_length = (total_length // context_length) * context_length
    
    # Split the massive list into chunks of context_length
    result = {
        "input_ids": [
            concatenated_ids[i : i + context_length]
            for i in range(0, total_length, context_length)
        ]
    }
    
    # For language modeling, the labels are the exact same as the inputs
    # (The transformer will automatically shift them by 1 internally to predict the "next" token)
    result["labels"] = result["input_ids"].copy()
    return result

print("Chunking tokens into fixed sizes...")
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=NUM_PROC,
    fn_kwargs={"context_length": CONTEXT_LENGTH} 
)

print(lm_datasets)